# Matching Engine

Continues from `skill_extraction_optimized.ipynb`. This notebook:
1. Loads postings and resumes with their extracted skills
2. Computes keyword-based similarity (Jaccard) between skill sets
3. Computes semantic similarity (embeddings) between full text
4. Combines both into a final match score
5. Builds a gap analysis (missing skills)
6. Tests ranking on real data — pick one resume, rank all jobs by fit

## 1. Load data with extracted skills

The `skills` column was saved as a string representation of a Python set (e.g. `"{'python', 'sql'}"`) when written to CSV, so it needs to be converted back to an actual set using `ast.literal_eval`.

In [1]:
import pandas as pd
import ast

ds_postings = pd.read_csv(r"C:\Users\dubey\1-Code\Resume-Job Matching NLP Tool\data\ds_postings_with_skills.csv")
ds_resumes = pd.read_csv(r"C:\Users\dubey\1-Code\Resume-Job Matching NLP Tool\data\ds_resumes_with_skills.csv")

# Convert skills column from string back to a real Python set
ds_postings["skills"] = ds_postings["skills"].apply(ast.literal_eval)
ds_resumes["skills"] = ds_resumes["skills"].apply(ast.literal_eval)

print("Postings:", ds_postings.shape)
print("Resumes: ", ds_resumes.shape)
print("Example postings skills:", ds_postings["skills"].iloc[0])
print("Example resume skills:  ", ds_resumes["skills"].iloc[0])

Postings: (1187, 8)
Resumes:  (238, 6)
Example postings skills: {'sql', 'kafka', 'etl', 'snowflake', 'optimization', 'python', 'linux'}
Example resume skills:   {'linux', 'html', 'teamwork', 'big data'}


## 2. Keyword-based matching (Jaccard similarity)

`jaccard = |intersection| / |union|` — a value between 0 and 1 showing how much two skill sets overlap, regardless of how large each set is. Fast, interpretable, and a solid baseline.

In [2]:
def jaccard_similarity(skills_a: set, skills_b: set) -> float:
    if not skills_a or not skills_b:
        return 0.0
    intersection = len(skills_a & skills_b)
    union = len(skills_a | skills_b)
    return intersection / union if union > 0 else 0.0

# Quick test on the first resume and first posting
test_score = jaccard_similarity(ds_resumes["skills"].iloc[0], ds_postings["skills"].iloc[0])
print("Jaccard similarity (sample):", round(test_score, 3))

Jaccard similarity (sample): 0.1


## 3. Semantic matching (sentence embeddings)

Uses `all-MiniLM-L6-v2` from sentence-transformers to embed full resume/job text into vectors, then compares them with cosine similarity. This catches matches keyword overlap misses — e.g. "built data pipelines" vs "ETL development" have zero shared keywords but mean nearly the same thing.

First run will download the model (~80MB) — needs internet access once, then it's cached locally.

In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded.")

c:\Users\dubey\anaconda3\envs\job\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2483.06it/s]


Model loaded.


## 4. Pre-compute embeddings for all postings and resumes

This is the slow step (a few minutes depending on your machine) — but it only needs to run once. Embeddings are stored as numpy arrays so every resume-job comparison afterward is just a fast vector operation.

In [4]:
posting_embeddings = model.encode(
    ds_postings["description_clean"].tolist(),
    show_progress_bar=True,
    batch_size=32
)

resume_embeddings = model.encode(
    ds_resumes["Resume_clean"].tolist(),
    show_progress_bar=True,
    batch_size=32
)

print("Posting embeddings shape:", posting_embeddings.shape)
print("Resume embeddings shape: ", resume_embeddings.shape)

# Save so this expensive step never needs to be repeated
np.save(r"C:\Users\dubey\1-Code\Resume-Job Matching NLP Tool\data\posting_embeddings.npy", posting_embeddings)
np.save(r"C:\Users\dubey\1-Code\Resume-Job Matching NLP Tool\data\resume_embeddings.npy", resume_embeddings)

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Batches: 100%|██████████| 8/8 [00:08<00:00,  1.03s/it]

Posting embeddings shape: (1187, 384)
Resume embeddings shape:  (238, 384)


## 5. Combine keyword + semantic scores into a final match function

Weights (0.4 keyword / 0.6 semantic) are a reasonable starting point — semantic similarity tends to be more informative since it understands meaning, not just exact words, but keyword overlap adds interpretability. Adjust after seeing results on real pairs.

In [5]:
def match_score(resume_idx: int, posting_idx: int, keyword_weight: float = 0.4) -> dict:
    resume_skills = ds_resumes["skills"].iloc[resume_idx]
    posting_skills = ds_postings["skills"].iloc[posting_idx]

    jaccard = jaccard_similarity(resume_skills, posting_skills)

    resume_vec = resume_embeddings[resume_idx].reshape(1, -1)
    posting_vec = posting_embeddings[posting_idx].reshape(1, -1)
    semantic = cosine_similarity(resume_vec, posting_vec)[0][0]

    final = keyword_weight * jaccard + (1 - keyword_weight) * semantic

    return {
        "jaccard": round(jaccard, 3),
        "semantic": round(float(semantic), 3),
        "final_score": round(float(final), 3),
        "matched_skills": resume_skills & posting_skills,
        "missing_skills": posting_skills - resume_skills
    }

# Test on the first resume and first posting
result = match_score(0, 0)
print("Job title:", ds_postings["title"].iloc[0])
print("Resume category:", ds_resumes["Category"].iloc[0])
print(result)

Job title: Sr Data Engineer with Kafka
Resume category: INFORMATION-TECHNOLOGY
{'jaccard': 0.1, 'semantic': 0.386, 'final_score': 0.271, 'matched_skills': {'linux'}, 'missing_skills': {'sql', 'kafka', 'etl', 'snowflake', 'optimization', 'python'}}


## 6. Rank all job postings for one resume

This is the real end-to-end test: pick a resume, score it against every job posting, and see if the top matches actually make sense.

In [6]:
def rank_jobs_for_resume(resume_idx: int, top_n: int = 10) -> pd.DataFrame:
    results = []
    for posting_idx in range(len(ds_postings)):
        r = match_score(resume_idx, posting_idx)
        results.append({
            "title": ds_postings["title"].iloc[posting_idx],
            "company": ds_postings["company_name"].iloc[posting_idx],
            "final_score": r["final_score"],
            "jaccard": r["jaccard"],
            "semantic": r["semantic"],
            "matched_skills": r["matched_skills"],
            "missing_skills": r["missing_skills"]
        })
    ranked = pd.DataFrame(results).sort_values("final_score", ascending=False)
    return ranked.head(top_n)

# Test on the first resume in your filtered set
print("Resume category:", ds_resumes["Category"].iloc[0])
top_matches = rank_jobs_for_resume(resume_idx=0, top_n=10)
top_matches[["title", "company", "final_score", "jaccard", "semantic"]]

Resume category: INFORMATION-TECHNOLOGY


,title,company,final_score,jaccard,semantic
1047,Data Engineer,"DMI (Digital Management, LLC)",0.387,0.200,0.512
834,"Data Engineer - Enterprise Data and Analytics,...",Principal Financial Group,0.374,0.083,0.568
948,Data Analyst with Security Clearance,ClearanceJobs,0.371,0.000,0.619
961,Sr Data Engineer,Apex Systems,0.369,0.111,0.541
1107,IT Sr Data Engineer Delivery,Fulton Bank,0.367,0.125,0.528
107,"Senior Data Engineer (Java, Spring boot, AWS)",Fidelity Investments,0.365,0.182,0.487
7,eCommerce Data Analyst,Radiant Systems Inc,0.365,0.100,0.542
96,Senior Data Engineer,Revature,0.360,0.167,0.488
609,Senior Data Scientist,Two Six Technologies,0.359,0.167,0.486
574,Senior Data Engineer,Oracle,0.358,0.143,0.502


## 7. Inspect the top match in detail

Shows matched and missing skills for the single best match — this is the actual user-facing output your Streamlit app will eventually display.

In [7]:
best = top_matches.iloc[0]
print("Best match:", best["title"], "at", best["company"])
print("Final score:", best["final_score"])
print("Matched skills:", best["matched_skills"])
print("Missing skills:", best["missing_skills"])

Best match: Data Engineer at DMI (Digital Management, LLC)
Final score: 0.387
Matched skills: {'big data'}
Missing skills: {'data visualization'}


In [8]:
for i in [5, 50, 100]:
    print(f"Resume category: {ds_resumes['Category'].iloc[i]}")
    top = rank_jobs_for_resume(resume_idx=i, top_n=3)
    print(top[["title", "final_score"]])
    print()

Resume category: INFORMATION-TECHNOLOGY
                                title  final_score
176  Lead Data Engineer - Databricks         0.669
112         Machine Learning Engineer        0.663
19                Design Data Analyst        0.640

Resume category: INFORMATION-TECHNOLOGY
                                  title  final_score
485  Full Stack Developer/Data Engineer        0.556
779                        Data Analyst        0.541
7                eCommerce Data Analyst        0.538

Resume category: INFORMATION-TECHNOLOGY
                      title  final_score
7    eCommerce Data Analyst        0.527
344            Data Analyst        0.461
513            Data Analyst        0.449

